In [ ]:
import os
import glob
import cv2
import numpy as np
import tensorflow as tf
from sklearn.utils import shuffle
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, Dense, Dropout, Activation,
    GlobalAveragePooling2D, BatchNormalization,
    MaxPooling2D, Input, Flatten
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# Configuration
# ----------------------------
IMG_SIZE     = 100
DATASET_PATH = r"D:\Coding\PDL\dataset"

# Folder structure from GitHub:
#   dataset/
#     train_pothole/   train_normal/
#     test/pothole/    test/normal/

TRAIN_POTHOLE = os.path.join(DATASET_PATH, "train_pothole")
TRAIN_NORMAL  = os.path.join(DATASET_PATH, "train_normal")
TEST_POTHOLE  = os.path.join(DATASET_PATH, "test", "pothole")
TEST_NORMAL   = os.path.join(DATASET_PATH, "test", "normal")

# ----------------------------
# Utility: Load Images Safely
# ----------------------------
def load_images(folder_path):
    images = []
    for ext in ["jpg", "jpeg", "png", "JPG", "PNG"]:
        for img_path in glob.glob(os.path.join(folder_path, f"*.{ext}")):
            img = cv2.imread(img_path, 0)          # grayscale
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
    return np.asarray(images)

# ----------------------------
# Feature Extraction for Severity
# Extracts hand-crafted features that reflect damage:
#   - mean / std of pixel intensity (dark pits → low mean)
#   - edge density via Canny (rough potholes → more edges)
#   - low-frequency energy ratio (smooth road → high ratio)
# ----------------------------
def extract_severity_features(images):
    features = []
    for img in images:
        # Pixel statistics
        mean_val  = np.mean(img)
        std_val   = np.std(img)

        # Edge density (Canny)
        edges      = cv2.Canny(img, 50, 150)
        edge_dens  = np.sum(edges > 0) / edges.size

        # Laplacian variance (texture roughness)
        lap_var    = cv2.Laplacian(img.astype(np.float64), cv2.CV_64F).var()

        # Dark-region ratio (pothole pits tend to be darker)
        dark_ratio = np.sum(img < 80) / img.size

        features.append([mean_val, std_val, edge_dens, lap_var, dark_ratio])
    return np.array(features)

# ----------------------------
# Severity Labelling via K-Means
# Clusters pothole images into 3 groups, then ranks them
# by severity score (low mean + high edges = severe)
# ----------------------------
def assign_severity_labels(images):
    feats  = extract_severity_features(images)
    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(feats)

    km = KMeans(n_clusters=3, random_state=42, n_init=10)
    cluster_ids = km.fit_predict(feats_scaled)

    # Compute a severity score per cluster centre:
    #   higher edge density & lower mean → more severe
    centres = km.cluster_centers_                  # shape (3, 5)
    # Score = -mean_col + edge_dens_col + lap_var_col + dark_ratio_col
    #         (all already standardised, so comparable)
    severity_score = (
        -centres[:, 0]      # low intensity  → severe
        + centres[:, 2]     # edge density   → severe
        + centres[:, 3]     # laplacian var  → severe
        + centres[:, 4]     # dark ratio     → severe
    )
    # Map cluster → severity rank: 0=minor, 1=moderate, 2=severe
    rank_order = np.argsort(severity_score)        # ascending
    cluster_to_severity = {rank_order[i]: i for i in range(3)}

    severity_labels = np.array([cluster_to_severity[c] for c in cluster_ids])
    return severity_labels, cluster_ids

# -------------------------------------------------------
# Stage 1 – Binary CNN  (pothole vs normal)
# -------------------------------------------------------
def build_binary_cnn():
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), padding="same", input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        BatchNormalization(),
        Activation("relu"),
        Conv2D(32, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Block 2
        Conv2D(64, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        Conv2D(64, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Block 3
        Conv2D(128, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        GlobalAveragePooling2D(),
        Dropout(0.4),

        # Head
        Dense(256),
        Activation("relu"),
        Dropout(0.4),
        Dense(2),
        Activation("softmax"),
    ])
    return model

# -------------------------------------------------------
# Stage 2 – Severity CNN  (minor / moderate / severe)
# -------------------------------------------------------
def build_severity_cnn():
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), padding="same", input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        BatchNormalization(),
        Activation("relu"),
        Conv2D(32, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Block 2
        Conv2D(64, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        Conv2D(64, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        # Block 3
        Conv2D(128, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        Conv2D(128, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        GlobalAveragePooling2D(),
        Dropout(0.4),

        # Head
        Dense(256),
        Activation("relu"),
        Dropout(0.4),
        Dense(3),
        Activation("softmax"),
    ])
    return model

# ----------------------------
# Callbacks
# ----------------------------
def get_callbacks(monitor="val_loss"):
    return [
        EarlyStopping(monitor=monitor, patience=7, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor=monitor, factor=0.5, patience=3, verbose=1, min_lr=1e-6),
    ]

# ----------------------------
# Plot helpers
# ----------------------------
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["accuracy"],     label="train acc")
    axes[0].plot(history.history["val_accuracy"], label="val acc")
    axes[0].set_title(f"{title} – Accuracy"); axes[0].legend()
    axes[1].plot(history.history["loss"],     label="train loss")
    axes[1].plot(history.history["val_loss"], label="val loss")
    axes[1].set_title(f"{title} – Loss"); axes[1].legend()
    plt.tight_layout()
    safe = title.replace(" ", "_")
    plt.savefig(f"{safe}_training.png", dpi=120)
    plt.show()
    print(f"Saved: {safe}_training.png")

def plot_confusion(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(title); plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout()
    safe = title.replace(" ", "_")
    plt.savefig(f"{safe}_confusion.png", dpi=120)
    plt.show()
    print(f"Saved: {safe}_confusion.png")

# ================================================================
#  MAIN
# ================================================================
print("=" * 55)
print("  Pothole Detection + Severity Classification Pipeline")
print("=" * 55)

# ── Load raw images ──────────────────────────────────────────────
pothole_train = load_images(TRAIN_POTHOLE)
normal_train  = load_images(TRAIN_NORMAL)
pothole_test  = load_images(TEST_POTHOLE)
normal_test   = load_images(TEST_NORMAL)

print(f"\nDataset summary:")
print(f"  Train pothole : {len(pothole_train)}")
print(f"  Train normal  : {len(normal_train)}")
print(f"  Test  pothole : {len(pothole_test)}")
print(f"  Test  normal  : {len(normal_test)}")

if len(pothole_train) == 0 or len(normal_train) == 0:
    raise ValueError("Training images missing – check folder paths above.")

# ================================================================
#  STAGE 1 – Binary classification (pothole vs normal)
# ================================================================
print("\n" + "─" * 55)
print("  STAGE 1 – Binary CNN: pothole vs normal")
print("─" * 55)

X_tr_bin = np.concatenate([pothole_train, normal_train])
y_tr_bin = np.concatenate([np.ones(len(pothole_train)),
                            np.zeros(len(normal_train))])

X_te_bin = np.concatenate([pothole_test, normal_test])
y_te_bin = np.concatenate([np.ones(len(pothole_test)),
                            np.zeros(len(normal_test))])

X_tr_bin, y_tr_bin = shuffle(X_tr_bin, y_tr_bin, random_state=42)
X_te_bin, y_te_bin = shuffle(X_te_bin, y_te_bin, random_state=42)

X_tr_bin = X_tr_bin.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
X_te_bin = X_te_bin.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0

y_tr_bin_cat = to_categorical(y_tr_bin, 2)
y_te_bin_cat = to_categorical(y_te_bin, 2)

binary_model = build_binary_cnn()
binary_model.summary()

binary_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_bin = binary_model.fit(
    X_tr_bin, y_tr_bin_cat,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=get_callbacks(),
    verbose=1
)

plot_history(history_bin, "Stage1 Binary CNN")

bin_loss, bin_acc = binary_model.evaluate(X_te_bin, y_te_bin_cat, verbose=0)
print(f"\n[Stage 1] Test Accuracy: {bin_acc:.4f}  |  Loss: {bin_loss:.4f}")

y_pred_bin = np.argmax(binary_model.predict(X_te_bin), axis=1)
print("\nBinary Classification Report:")
print(classification_report(y_te_bin.astype(int), y_pred_bin,
                             target_names=["normal", "pothole"]))
plot_confusion(y_te_bin.astype(int), y_pred_bin,
               ["normal", "pothole"], "Stage1 Binary CNN")

binary_model.save("binary_pothole_cnn.keras")
print("Saved: binary_pothole_cnn.keras")

# ================================================================
#  STAGE 2 – Severity classification (minor / moderate / severe)
# ================================================================
print("\n" + "─" * 55)
print("  STAGE 2 – Severity CNN: minor / moderate / severe")
print("─" * 55)

# ── Assign pseudo-labels via K-Means clustering ──────────────────
print("\nClustering training potholes into 3 severity groups …")
sev_labels_train, _ = assign_severity_labels(pothole_train)

counts = np.bincount(sev_labels_train)
label_names = ["minor", "moderate", "severe"]
for i, name in enumerate(label_names):
    print(f"  {name:10s}: {counts[i]} images")

# ── Prepare severity datasets ────────────────────────────────────
X_tr_sev = pothole_train.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
y_tr_sev  = to_categorical(sev_labels_train, 3)

X_tr_sev, y_tr_sev = shuffle(X_tr_sev, y_tr_sev, random_state=42)

# Label test potholes using same feature-based approach
print("Assigning severity labels to test potholes …")
sev_labels_test, _ = assign_severity_labels(pothole_test)
X_te_sev = pothole_test.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
y_te_sev  = to_categorical(sev_labels_test, 3)

# ── Train severity model ─────────────────────────────────────────
severity_model = build_severity_cnn()
severity_model.summary()

severity_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_sev = severity_model.fit(
    X_tr_sev, y_tr_sev,
    epochs=50,
    batch_size=32,
    validation_split=0.15,
    callbacks=get_callbacks(),
    verbose=1
)

plot_history(history_sev, "Stage2 Severity CNN")

sev_loss, sev_acc = severity_model.evaluate(X_te_sev, y_te_sev, verbose=0)
print(f"\n[Stage 2] Test Accuracy: {sev_acc:.4f}  |  Loss: {sev_loss:.4f}")

y_pred_sev = np.argmax(severity_model.predict(X_te_sev), axis=1)
print("\nSeverity Classification Report:")
print(classification_report(sev_labels_test, y_pred_sev,
                             target_names=label_names))
plot_confusion(sev_labels_test, y_pred_sev,
               label_names, "Stage2 Severity CNN")

severity_model.save("severity_pothole_cnn.keras")
print("Saved: severity_pothole_cnn.keras")

# ================================================================
#  INFERENCE HELPER  (single image end-to-end pipeline)
# ================================================================
def predict_single_image(img_path: str) -> dict:
    """
    Given an image path, returns:
      { "detection": "pothole" | "normal",
        "severity" : "minor" | "moderate" | "severe" | "N/A",
        "binary_confidence": float,
        "severity_confidence": float | None }
    """
    img = cv2.imread(img_path, 0)
    if img is None:
        raise FileNotFoundError(f"Cannot read: {img_path}")
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    tensor = img.reshape(1, IMG_SIZE, IMG_SIZE, 1) / 255.0

    # Stage 1
    bin_probs  = binary_model.predict(tensor, verbose=0)[0]
    is_pothole = int(np.argmax(bin_probs))
    bin_conf   = float(bin_probs[is_pothole])

    if is_pothole == 0:
        return {"detection": "normal", "severity": "N/A",
                "binary_confidence": bin_conf, "severity_confidence": None}

    # Stage 2
    sev_probs = severity_model.predict(tensor, verbose=0)[0]
    sev_idx   = int(np.argmax(sev_probs))
    sev_conf  = float(sev_probs[sev_idx])

    return {
        "detection"           : "pothole",
        "severity"            : label_names[sev_idx],
        "binary_confidence"   : bin_conf,
        "severity_confidence" : sev_conf
    }

print("\n" + "=" * 55)
print("  Pipeline complete!")
print("  Saved models: binary_pothole_cnn.keras")
print("                severity_pothole_cnn.keras")
print("=" * 55)
print("\nExample usage of predict_single_image():")
print('  result = predict_single_image(r"D:\\path\\to\\image.jpg")')
print('  print(result)')
print('  # → {"detection": "pothole", "severity": "severe", ...}')

Working directory: D:\Coding\ML
Dataset exists: True
Pothole train images: 618
Normal train images : 664
Pothole test images : 40
Normal test images  : 40
Training data shape: (1282, 100, 100, 1)
Training labels shape: (1282, 2)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)                    │ (None, 24, 24, 16)          │           1,040 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_8 (Activation)            │ (None, 24, 24, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 24, 24, 32)          │          12,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_9 (Activation)            │ (None, 24, 24, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 32)                  │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 512)                 │          16,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_10 (Activation)           │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 2)                   │           1,026 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_11 (Activation)           │ (None, 2)                   │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 31,794 (124.20 KB)

 Trainable params: 31,794 (124.20 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.5153 - loss: 0.6943 - val_accuracy: 0.4806 - val_loss: 0.6935
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5238 - loss: 0.6922 - val_accuracy: 0.4806 - val_loss: 0.6932
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5182 - loss: 0.6926 - val_accuracy: 0.4884 - val_loss: 0.6926
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5297 - loss: 0.6923 - val_accuracy: 0.4806 - val_loss: 0.6941
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5140 - loss: 0.6935 - val_accuracy: 0.4806 - val_loss: 0.6981
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5131 - loss: 0.6940 - val_accuracy: 0.4806 - val_loss: 0.6967
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5231 - loss: 0.6920 - val_accuracy: 0.4806 - val_loss: 0.6943
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5099 - loss: 0.6920 - val_accuracy: 0.4806 - v

Test Accuracy: 0.5249999761581421
Model saved as pothole_cnn.h5
